# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [ ]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [ ]:
df.head()

## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [49]:
from nltk.stem import RSLPStemmer

stemmer = RSLPStemmer()

# Negations carry the meaning in a bad review ("não recomendo" = do not recommend),
# so they are deliberately kept out of the stopword list.
NEGATIONS = {"não", "nao", "nem", "nunca", "sem"}

_pt = set(stopwords.words("portuguese"))
_pt_ascii = {unidecode.unidecode(w) for w in _pt}
STOPWORDS_PT = (_pt | _pt_ascii) - NEGATIONS


def tokenize_pt(sentence):
    """Lowercase, drop digits/punctuation, tokenize, remove stopwords.
    Accents are kept here on purpose: the RSLP rules are written for accented text."""
    sentence = str(sentence).lower().strip()
    sentence = "".join(ch for ch in sentence if not ch.isdigit())
    sentence = "".join(" " if ch in string.punctuation else ch for ch in sentence)
    tokens = word_tokenize(sentence, language="portuguese")
    return [t for t in tokens if t not in STOPWORDS_PT and len(t) > 2]


def cleaning(sentence):
    """Baseline track: readable Portuguese words, accents folded to ASCII."""
    return " ".join(unidecode.unidecode(t) for t in tokenize_pt(sentence))


def cleaning_stemmed(sentence):
    """Stemmed track: stem first (rules need accents), then fold to ASCII."""
    return " ".join(unidecode.unidecode(stemmer.stem(t)) for t in tokenize_pt(sentence))


# Sanity check on one review
sample = df.loc[0, "full_review"]
print("RAW:", sample[:160])
print("CLEANED:", cleaning(sample)[:160])
print("STEMMED:", cleaning_stemmed(sample)[:160])

RAW:  Recebi bem antes do prazo estipulado.
CLEANED: recebi bem antes prazo estipulado
STEMMED: receb bem ant praz estipul


In [ ]:
df["full_review_cleaned"] = df["full_review"].apply(cleaning)
df["full_review_stemmed"] = df["full_review"].apply(cleaning_stemmed)

# How much does stemming actually collapse the vocabulary?
vocab_cleaned = set(" ".join(df["full_review_cleaned"]).split())
vocab_stemmed = set(" ".join(df["full_review_stemmed"]).split())

print(f"vocabulary — cleaned: {len(vocab_cleaned):,}")
print(f"vocabulary — stemmed: {len(vocab_stemmed):,}")
print(f"reduction: {1 - len(vocab_stemmed) / len(vocab_cleaned):.1%}")

df[["review_score", "full_review", "full_review_cleaned", "full_review_stemmed"]].head()

## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [ ]:
bad_mask = df["review_score"].isin([1, 2, 3])

n_bad = bad_mask.sum()
share_bad = bad_mask.mean()

print(f"bad reviews (1-3): {n_bad:,} / {len(df):,} → {share_bad:.1%}")
print()
print(df["review_score"].value_counts(normalize=True).sort_index().mul(100).round(1))

🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [ ]:
bad = df[bad_mask].copy().reset_index(drop=True)

print(f"shape: {bad.shape}")
print(f"mean review length: {bad['length_review'].mean():.0f} chars "
      f"(vs {df.loc[~bad_mask, 'length_review'].mean():.0f} for 4-5 star)")
print()
print("top categories among bad reviews:")
print(bad["product_category_name"].value_counts().head(10))

bad[["review_score", "product_category_name", "full_review_cleaned"]].head()

### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [ ]:
VECT_PARAMS = dict(
    ngram_range=(1, 2),   # unigrams + bigrams ("nao recomendo")
    max_df=0.75,          # drop terms appearing in >75% of reviews
    max_features=5000,
)

# Track A: readable words
count_vectorizer = CountVectorizer(**VECT_PARAMS)
X_bad = count_vectorizer.fit_transform(bad["full_review_cleaned"])

# Track B: stemmed
count_vectorizer_stem = CountVectorizer(**VECT_PARAMS)
X_bad_stem = count_vectorizer_stem.fit_transform(bad["full_review_stemmed"])

print(f"cleaned  → {X_bad.shape[0]:,} docs × {X_bad.shape[1]:,} features")
print(f"stemmed  → {X_bad_stem.shape[0]:,} docs × {X_bad_stem.shape[1]:,} features")

# How many of the 5000 selected features are actually bigrams?
for name, vec in [("cleaned", count_vectorizer), ("stemmed", count_vectorizer_stem)]:
    feats = vec.get_feature_names_out()
    n_bi = sum(" " in f for f in feats)
    print(f"{name:8} bigrams: {n_bi:,} / {len(feats):,} ({n_bi/len(feats):.0%})")

freq = pd.DataFrame({
    "term": count_vectorizer.get_feature_names_out(),
    "count": X_bad.sum(axis=0).A1,
}).sort_values("count", ascending=False)

print("\ntop 15 unigrams:")
print(freq[~freq["term"].str.contains(" ")].head(15).to_string(index=False))

print("\ntop 15 bigrams:")
print(freq[freq["term"].str.contains(" ")].head(15).to_string(index=False))

### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [ ]:
LDA_PARAMS = dict(n_components=3, random_state=42, learning_method="batch")

lda = LatentDirichletAllocation(**LDA_PARAMS)
lda.fit(X_bad)

lda_stem = LatentDirichletAllocation(**LDA_PARAMS)
lda_stem.fit(X_bad_stem)

print("cleaned  components_:", lda.components_.shape)
print("stemmed  components_:", lda_stem.components_.shape)
print("perplexity — cleaned:", round(lda.perplexity(X_bad), 1))
print("perplexity — stemmed:", round(lda_stem.perplexity(X_bad_stem), 1))

#### Belge Karışımı (Konular için)

In [ ]:
document_topic_mixture = lda.transform(X_bad)

print("shape:", document_topic_mixture.shape)
pd.DataFrame(
    document_topic_mixture,
    columns=[f"topic_{i}" for i in range(3)],
).head()

👉 Her inceleme için en önemli konuyu rapor edelim

In [ ]:
bad["most_important_topic"] = document_topic_mixture.argmax(axis=1)

print(bad["most_important_topic"].value_counts().sort_index())
print()
print("mean review score per topic:")
print(bad.groupby("most_important_topic")["review_score"].agg(["count", "mean"]).round(2))

# Propagate topic assignment back to the full dataframe (A plan).
# Reviews scored 4-5 get transformed with the same fitted model, so every
# row has a topic; section (4) then filters down to the worst categories anyway.
X_all = count_vectorizer.transform(df["full_review_cleaned"])
df["most_important_topic"] = lda.transform(X_all).argmax(axis=1)

print("\nfull dataframe topic distribution:")
print(df["most_important_topic"].value_counts().sort_index())

#### Konu Karışımı (Kelimeler için)

In [ ]:
topic_word_mixture = lda.components_

print("shape:", topic_word_mixture.shape)

pd.DataFrame(
    topic_word_mixture,
    index=[f"topic_{i}" for i in range(3)],
    columns=count_vectorizer.get_feature_names_out(),
).iloc[:, :8]

#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [ ]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [ ]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


### 🔍 Konuların Yorumlanması

LDA, kötü yorumlar (1–3 yıldız, n=9.658) üzerinde üç ayrı şikâyet ekseni buldu:

| Konu | Tema | Belirleyici kelimeler | Yorum sayısı | Ort. puan |
|---|---|---|---|---|
| 0 | **Ürün beklentiyi karşılamadı** | `diferente`, `foto`, `defeito`, `troca`, `produto veio` | 3.738 | 1.73 |
| 1 | **Teslimat & lojistik (ılımlı)** | `entrega`, `prazo`, `frete`, `correios`, `porem` | 2.701 | 2.30 |
| 2 | **Ürün hiç gelmedi / eksik geldi** | `nao recebi`, `apenas`, `ainda`, `dois`, `pedido` | 3.219 | 1.52 |

**Konu 1'de neden olumlu kelimeler var?** `bom`, `bem` ve `antes` kelimelerinin varlığı çelişki değil, kalıbın kendisi: 3 yıldızlı yorumlar tipik olarak *"ürün iyi **ancak** kargo geç geldi"* biçiminde yazılıyor. `porem` (= ancak) kelimesinin ilk sıralarda olması bunu doğruluyor. Ortalama puanın 2.30 ile en yüksek olması da aynı yönde.

**En sert eksen hangisi?** Konu 2 (ort. 1.52). Yani Olist'te müşteriyi en çok kızdıran şey ürünün kötü olması değil, **ürünün hiç ulaşmaması veya eksik ulaşması**. Ham frekanslar da bunu destekliyor: en sık bigram `nao recebi` (742 kez), ardından `recebi apenas` (242) ve `nao entregue` (230).

**Stemli hat ile doğrulama.** Aynı LDA, RSLP ile köklenmiş metinlere ayrıca uygulandı (kelime dağarcığı 12.827 → 6.873, %46,4 azalma). Konu numaraları farklı sırada çıktı ama **üç eksen birebir yeniden üretildi** (cleaned 0 ↔ stemmed 1, cleaned 1 ↔ stemmed 0, cleaned 2 ↔ stemmed 2). Bu, konuların vektörleştirme tercihine bağlı bir tesadüf olmadığını gösteriyor. Stemli hattın tek ek katkısı Konu 2'de yüzeye çıkan `falt` (*falta* = eksik) kökü oldu.

> ⚠️ İki hattın perplexity değerleri (1.131,8 ve 1.046,7) **karşılaştırılabilir değildir**, çünkü farklı kelime uzaylarında hesaplanmıştır. Stemli hattın düşük değeri kalite üstünlüğü olarak okunmamalıdır.

**Metodolojik not.** LDA yalnızca kötü yorumlar üzerinde `fit` edildi; konular bu yorumlardan öğrenildi. Tüm veri kümesine ise `transform` uygulanarak konu ataması yayıldı, böylece (4). bölümdeki kategori bazlı gruplamalar tam veri üzerinde çalışabiliyor.

🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [ ]:
print("=" * 22, "CLEANED", "=" * 22)
print_topics(count_vectorizer, lda, 12)

print()
print("=" * 22, "STEMMED", "=" * 22)
print_topics(count_vectorizer_stem, lda_stem, 12)

🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [ ]:
# Redefines topic_word_mixture: the cells below (the `most_important_words`
# assignment and `bad_reviews_seller`) index it as a list of word-lists,
# not as the raw lda.components_ weight matrix defined earlier.
topic_word_mixture = [
    topic_word(count_vectorizer, lda, topic, 10, with_weights=False)
    for topic in lda.components_
]

TOPIC_LABELS = {
    0: "Ürün beklentiyi karşılamadı (farklı/defolu/foto uyumsuz)",
    1: "Teslimat & lojistik (ılımlı, çoğunlukla 3 yıldız)",
    2: "Ürün hiç gelmedi veya eksik geldi",
}

for i, words in enumerate(topic_word_mixture):
    print(f"topic {i} — {TOPIC_LABELS[i]}")
    print(f"  {words}\n")

In [ ]:
df["most_important_words"] = df["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [ ]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

## (3.4) Pipeline Tf-Idf ve LDA

In [ ]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [ ]:
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    TfidfVectorizer(**VECT_PARAMS),
    LatentDirichletAllocation(**LDA_PARAMS),
)

pipeline.fit(bad["full_review_cleaned"])
pipeline

💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [ ]:
pipeline._final_estimator

In [ ]:
pipeline._final_estimator.components_

Pipeline ile **Belge Karışımı**:

In [ ]:
document_topic_mixture_pipe = pipeline.transform(bad["full_review_cleaned"])

print("shape:", document_topic_mixture_pipe.shape)
pd.DataFrame(
    document_topic_mixture_pipe,
    columns=[f"topic_{i}" for i in range(3)],
).head()

Pipeline ile **Konu Karışımı**:

In [ ]:
pipe_vectorizer = pipeline.named_steps["tfidfvectorizer"]
pipe_lda = pipeline._final_estimator

topic_word_mixture_pipe = pipe_lda.components_
print("shape:", topic_word_mixture_pipe.shape)

print_topics(pipe_vectorizer, pipe_lda, 12)

# Does TF-IDF weighting reproduce the same three axes as raw counts?
for i in range(3):
    count_words = set(topic_word(count_vectorizer, lda, lda.components_[i], 10, with_weights=False))
    for j in range(3):
        pipe_words = set(topic_word(pipe_vectorizer, pipe_lda, pipe_lda.components_[j], 10, with_weights=False))
        overlap = count_words & pipe_words
        if len(overlap) >= 4:
            print(f"count topic {i}  ↔  tfidf topic {j}   ({len(overlap)}/10 shared: {sorted(overlap)})")

#### Pipeline doğrulaması

TF-IDF ağırlıklandırması, ham sayımlarla bulunan üç ekseni **aynı konu numaralarıyla** yeniden üretti:

| Eşleşme | Ortak kelime (ilk 10'da) |
|---|---|
| count 0 ↔ tfidf 0 | 7/10 |
| count 1 ↔ tfidf 1 | 10/10 |
| count 2 ↔ tfidf 2 | 8/10 |

Böylece konular üç bağımsız kurulumda doğrulanmış oldu: CountVectorizer, RSLP ile köklenmiş metin ve TF-IDF. Yapı, vektörleştirme tercihinden bağımsız.

TF-IDF'in katkısı, her yerde geçen terimleri bastırarak daha spesifik ifadeleri öne çıkarması oldu: Konu 0'da `nao gostei` ve `quero` (*quero trocar/devolver* — değiştirmek/iade etmek istiyorum), Konu 1'de ise `demorou` (= gecikti).

## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [ ]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [ ]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [ ]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [ ]:
worst_products_reviews["most_important_topic"].value_counts()

In [ ]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

In [ ]:
[topic_word_mixture[i] for i in bad_frequency]

In [ ]:
# The `worst_products_reviews` cell above filters by category only, so it still
# contains 4-5 star reviews. Restrict to actual bad reviews before reading topics.
wp_bad = worst_products_reviews[worst_products_reviews["review_score"].isin([1, 2, 3])]

print(f"{len(wp_bad):,} bad reviews out of {len(worst_products_reviews):,} rows\n")

print("topic distribution — bad reviews only:")
print(wp_bad["most_important_topic"].value_counts())

print("\nvs. all bad reviews (baseline):")
print(bad["most_important_topic"].value_counts(normalize=True).mul(100).round(1))
print("\nworst categories, bad reviews only:")
print(wp_bad["most_important_topic"].value_counts(normalize=True).mul(100).round(1))

In [ ]:
print(pd.crosstab(
    wp_bad["product_category_name"],
    wp_bad["most_important_topic"],
    normalize="index",
).mul(100).round(1))

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [ ]:
import os
import sys
from pathlib import Path

# The olist helper package lives in a sibling repo (Workintech decision-science module).
# Override with the OLIST_PKG environment variable if it sits elsewhere.
OLIST_PKG = Path(os.environ.get("OLIST_PKG", Path.home() / "projects" / "data-context-and-setup"))
if not (OLIST_PKG / "olist").is_dir():
    raise FileNotFoundError(f"olist package not found at {OLIST_PKG} — set OLIST_PKG")
if str(OLIST_PKG) not in sys.path:
    sys.path.append(str(OLIST_PKG))

from olist.seller import Seller
from olist.product import Product
from olist.data import Olist

data = Olist().get_data()
sellers = Seller().get_training_data()
sellers.columns

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [ ]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

### (5.2) En kötü satıcılar tarafından satılan ürünler

In [ ]:
products = Product().get_training_data() [["product_id", "category"]]
products

❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [ ]:
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

In [ ]:
sellers_product_category.groupby("seller_id").count()

### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [ ]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [ ]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

In [ ]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [ ]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


In [ ]:
# The notebook's worst_sellers ranks by profits, which tracks volume more than
# quality, and bad_reviews_seller only searches within 5 categories. Redo both.

# (a) Rank by review_score among sellers with enough orders to be meaningful
seller_quality = sellers[sellers["n_orders"] >= 30].sort_values("review_score").head(10)
print(seller_quality[["seller_id", "n_orders", "review_score", "share_of_one_stars", "profits"]].to_string(index=False))

# (b) Search bad reviews across ALL categories, not just the worst 5
all_bad_sellers = bad.merge(data["order_items"], on="order_id", how="left")
print(f"\nbad reviews joined to sellers: {len(all_bad_sellers):,} rows")


def seller_topics(seller_id):
    temp = all_bad_sellers[all_bad_sellers["seller_id"] == seller_id]
    if len(temp) == 0:
        return None
    dist = temp["most_important_topic"].value_counts(normalize=True).mul(100).round(0)
    top = int(temp["most_important_topic"].value_counts().index[0])
    return len(temp), dict(dist), TOPIC_LABELS[top]

In [ ]:
for sid in seller_quality["seller_id"]:
    print("-" * 60)
    row = sellers[sellers.seller_id == sid].iloc[0]
    print(f"seller #{sid[:8]}  score={row.review_score:.2f}  orders={int(row.n_orders)}")
    print(focus_seller(sid).to_string())
    res = seller_topics(sid)
    if res:
        n, dist, label = res
        print(f"  {n} bad reviews — {dist}")
        print(f"  dominant: {label}")
    else:
        print("  no bad reviews found")

## 📌 Bulgular ve Metodolojik Notlar

### 1. Notebook'un "en kötü satıcılar" tanımı yanıltıcı

`worst_sellers` hücresi satıcıları **`profits`** değerine göre sıralıyor. Çıkan 10 satıcının puan ortalamaları 3,35 – 4,07 aralığında, yani platform ortalamasına yakın ve bir kısmı ortalamanın üzerinde. Bunlar *zarar ettiren* satıcılar; *kötü yorum alan* satıcılar değil.

Nedeni hacim: `cost_of_reviews` satış adediyle ölçeklendiği için en çok satan satıcı en çok zarar yazıyor. Listenin başındaki satıcı 1.624 adet saat/hediye ürünü satmış ve puanı 3,91.

**Düzeltme.** En az 30 siparişi olan satıcılar `review_score`'a göre sıralandığında **tamamen farklı bir liste** çıkıyor — iki liste arasında tek bir ortak satıcı yok:

| seller_id (ilk 8) | Sipariş | Ort. puan | 1 yıldız oranı |
|---|---|---|---|
| `1ca7077d` | 115 | 2,20 | **%58,8** |
| `2eb70248` | 202 | 2,71 | %38,8 |
| `602044f2` | 50 | 2,93 | %37,3 |
| `54965bbe` | 78 | 2,94 | %42,0 |
| `a49928bc` | 98 | 2,95 | %34,9 |

`1ca7077d` gerçek bir uç değer: her beş siparişten üçü 1 yıldız alıyor. Kâr sıralamasında bu satıcı hiç görünmüyor.

30 sipariş alt sınırı bilinçli — aksi halde 2 sipariş alıp ikisinden de 1 yıldız yiyen satıcılar listeyi doldurur ve analiz anlamsızlaşır.

### 2. `bad_reviews_seller` fonksiyonu satıcıların çoğunu göremiyor

Notebook'un verdiği yardımcı fonksiyon yalnızca `worst_products_reviews` içinde arama yapıyor; bu da yalnızca 5 kötü kategoriye filtrelenmiş 968 satır. Sonuç: 10 satıcının **6'sı için `None`** dönüyor. Saat, kozmetik veya telefon satan bir satıcı bu 5 kategoride olmadığı için kötü yorumu yokmuş gibi görünüyor — oysa var.

**Düzeltme.** Kötü yorumların tamamı (n=9.658) `order_items` ile birleştirildiğinde her satıcı için konu dağılımı elde ediliyor.

> ⚠️ Birleştirme 13.370 satır üretiyor (9.658 yorumdan fazla), çünkü bir sipariş birden çok kalem içerebiliyor ve yorum her kalem için tekrarlanıyor. Satıcı bazlı sayımlar bu nedenle **kalem düzeyindedir**, yorum düzeyinde değil.

### 3. Kötü satıcıların iki ayrı başarısızlık modu var

Düzeltilmiş listede satıcılar net biçimde ikiye ayrılıyor:

**Ürün sorunu ağırlıklı (Konu 0):**
- `2eb70248` — %82 (saat/hediye + bilgisayar aksesuarı)
- `a49928bc` — %72 (oyuncak)
- `1ca7077d` — %63 (telefon + oyuncak)

**Teslimat sorunu ağırlıklı (Konu 2):**
- `602044f2` — %67 (yalnızca ev tekstili)
- `8e6d7754` — %58 (yalnızca bilgisayar aksesuarı)
- `bbad7e51` — %47 (bebek ürünleri ağırlıklı)

Bu ayrım operasyonel olarak anlamlı: ilk grup için ürün listeleme/kalite kontrol müdahalesi, ikinci grup için kargo ve stok süreci müdahalesi gerekiyor. Tek tip bir "kötü satıcı" politikası ikisine de uymaz.

### 4. Kötü kategorilerin de kendi başarısızlık modları var

`worst_products_reviews` içindeki 968 satırın yalnızca 414'ü gerçekten kötü yorum — kalanı aynı kategorilerdeki 4-5 yıldızlı yorumlar. Filtrelenmeden bakıldığında Konu 1 baskın görünüyor (535), ancak bu tamamen iyi yorumlardan kaynaklanan bir yanılsama.

Yalnızca kötü yorumlarda, kategori bazında konu dağılımı (%):

| Kategori | Konu 0 (ürün) | Konu 1 (lojistik) | Konu 2 (gelmedi) |
|---|---|---|---|
| `fashion_roupa_masculina` | **66,7** | 14,3 | 19,0 |
| `telefonia_fixa` | **47,1** | 26,5 | 26,5 |
| `moveis_escritorio` | 39,8 | 24,9 | 35,2 |
| `casa_construcao` | 33,3 | 18,2 | **48,5** |
| `climatizacao` | 21,9 | 31,2 | **46,9** |

- **Erkek giyim** neredeyse tamamen ürün kaynaklı: yanlış beden, fotoğraftan farklı.
- **Yapı malzemesi** ve **klima** teslimat kaynaklı — ikisi de hacimli/ağır ürün; büyük paket lojistiği başarısız oluyor.
- **Ofis mobilyası** en düşük ortalamaya sahip kategori (546 yorum, 3,30) ama baskın tek bir sorunu yok (39,8 / 24,9 / 35,2). Tek bir ekseni düzeltmek yetmez.

Genel kötü yorum dağılımıyla (%38,7 / %28,0 / %33,3) karşılaştırıldığında, kötü kategorilerin toplam sapması ılımlı. Asıl fark **kategoriler arasında**, kategori grubunun geneli ile ortalama arasında değil.

### 5. Genel sonuç

Olist'te müşteriyi en çok kızdıran şey ürün kalitesi değil, **ürünün hiç ulaşmaması veya eksik ulaşması**. Kanıtlar:

- Konu 2 en düşük ortalama puana sahip (1,52; Konu 0: 1,73; Konu 1: 2,30)
- En sık geçen bigram `nao recebi` (742 kez), ardından `recebi apenas` (242) ve `nao entregue` (230)
- Kötü yorumlar iyi yorumların iki katı uzunlukta (96 vs 51 karakter) — şikâyet eden müşteri açıklama yapıyor, memnun olan "ok" deyip geçiyor

🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.